In [0]:
df=spark.read.csv('/Volumes/workspace/my_data/mentor/sales-data-raw.csv',header=True,inferSchema=True)
display(df)

OrderID,Date,SalesPerson,Amount,Product,Region,Column1
1001,2023-01-05,Alice Smith,$1,000,Laptop,North
1002,1/6/2023,Bob Jones,$700,Phone,SOUTH,null
1003,2023-01-07,Carol Wang,null,Tablet,east,null
1004,07-01-2023,alice smith,1100,Laptop,North,null
1005,2023-01-08,Bob Jones,$700,Phone,SOUTH,null
1006,2023-01-08,null,850,Tablet,West,null
1006,2023-01-08,null,850,Tablet,West,null
1007,2023/01/08,Scott Rogers,$900,Phone,WEST,null
1008,8-Jan-2023,Linda Moore,$1,250,Laptop,North
1009,2023-01-10,James Wood,"$1,200",Laptop,EAST,null


In [0]:
df.show()
df.printSchema()
df.columns
df.count()

+-------+----------+------------+------+-------+------+-------+
|OrderID|      Date| SalesPerson|Amount|Product|Region|Column1|
+-------+----------+------------+------+-------+------+-------+
|   1001|2023-01-05| Alice Smith|    $1|    000|Laptop|  North|
|   1002|  1/6/2023|   Bob Jones|  $700|  Phone| SOUTH|   NULL|
|   1003|2023-01-07|  Carol Wang|  NULL| Tablet|  east|   NULL|
|   1004|07-01-2023| alice smith|  1100| Laptop| North|   NULL|
|   1005|2023-01-08|   Bob Jones|  $700|  Phone| SOUTH|   NULL|
|   1006|2023-01-08|        NULL|   850| Tablet|  West|   NULL|
|   1006|2023-01-08|        NULL|   850| Tablet|  West|   NULL|
|   1007|2023/01/08|Scott Rogers|  $900|  Phone|  WEST|   NULL|
|   1008|8-Jan-2023| Linda Moore|    $1|    250|Laptop|  North|
|   1009|2023-01-10|  James Wood|$1,200| Laptop|  EAST|   NULL|
|   1010| 11/1/2023| alicia keys|  NULL|Tablet | south|   NULL|
|   1011|  20230112|   Bob Jones|  $700|  Phone| South|   NULL|
|   1012|15-01-2023|  Carol Wang|   820|

36

In [0]:
from pyspark.sql.functions import col
df.groupBy("OrderID") \
  .count() \
  .filter(col("count")>1) \
  .show()

+-------+-----+
|OrderID|count|
+-------+-----+
|   1026|    2|
|   1006|    2|
+-------+-----+



In [0]:
#to remove a duplicate record in OrderID
clean_df=df.dropDuplicates(["OrderID"])

In [0]:
from pyspark.sql.functions import col, try_to_date, coalesce

clean_df = df.withColumn(
    "Date",
    coalesce(
        try_to_date(col("Date"), "yyyy-MM-dd"),
        try_to_date(col("Date"), "M/d/yyyy"),
        try_to_date(col("Date"), "dd-MM-yyyy"),
        try_to_date(col("Date"), "yyyyMMdd"),
        try_to_date(col("Date"), "d-MMM-yyyy"),
        try_to_date(col("Date"), "dd/MM/yyyy")
    )
)



In [0]:
clean_df.orderBy("OrderID",descending=True).show()

+-------+----------+------------+------+-------+------+-------+
|OrderID|      Date| SalesPerson|Amount|Product|Region|Column1|
+-------+----------+------------+------+-------+------+-------+
|   1001|2023-01-05| Alice Smith|    $1|    000|Laptop|  North|
|   1002|2023-01-06|   Bob Jones|  $700|  Phone| SOUTH|   NULL|
|   1003|2023-01-07|  Carol Wang|  NULL| Tablet|  east|   NULL|
|   1004|2023-01-07| alice smith|  1100| Laptop| North|   NULL|
|   1005|2023-01-08|   Bob Jones|  $700|  Phone| SOUTH|   NULL|
|   1006|2023-01-08|        NULL|   850| Tablet|  West|   NULL|
|   1006|2023-01-08|        NULL|   850| Tablet|  West|   NULL|
|   1007|      NULL|Scott Rogers|  $900|  Phone|  WEST|   NULL|
|   1008|2023-01-08| Linda Moore|    $1|    250|Laptop|  North|
|   1009|2023-01-10|  James Wood|$1,200| Laptop|  EAST|   NULL|
|   1010|2023-11-01| alicia keys|  NULL|Tablet | south|   NULL|
|   1011|2023-01-12|   Bob Jones|  $700|  Phone| South|   NULL|
|   1012|2023-01-15|  Carol Wang|   820|

In [0]:
from pyspark.sql.functions import regexp_replace,col
clean_df=clean_df.withColumn(
    "Amount"
    ,regexp_replace(col("Amount"),"[$,]","").cast("double")
)

In [0]:
display(clean_df)

OrderID,Date,SalesPerson,Amount,Product,Region,Column1
1001,2023-01-05,Alice Smith,1.0,000,Laptop,North
1002,2023-01-06,Bob Jones,700.0,Phone,SOUTH,null
1003,2023-01-07,Carol Wang,null,Tablet,east,null
1004,2023-01-07,alice smith,1100.0,Laptop,North,null
1005,2023-01-08,Bob Jones,700.0,Phone,SOUTH,null
1006,2023-01-08,null,850.0,Tablet,West,null
1006,2023-01-08,null,850.0,Tablet,West,null
1007,null,Scott Rogers,900.0,Phone,WEST,null
1008,2023-01-08,Linda Moore,1.0,250,Laptop,North
1009,2023-01-10,James Wood,1200.0,Laptop,EAST,null


In [0]:
from pyspark.sql.functions import col,initcap,trim
clean_df=clean_df.withColumn(
    "SalesPerson",
    initcap(trim(col('SalesPerson'))).cast("string")
)

In [0]:
display(clean_df)

OrderID,Date,SalesPerson,Amount,Product,Region,Column1
1001,2023-01-05,Alice Smith,1.0,000,Laptop,North
1002,2023-01-06,Bob Jones,700.0,Phone,SOUTH,null
1003,2023-01-07,Carol Wang,null,Tablet,east,null
1004,2023-01-07,Alice Smith,1100.0,Laptop,North,null
1005,2023-01-08,Bob Jones,700.0,Phone,SOUTH,null
1006,2023-01-08,null,850.0,Tablet,West,null
1006,2023-01-08,null,850.0,Tablet,West,null
1007,null,Scott Rogers,900.0,Phone,WEST,null
1008,2023-01-08,Linda Moore,1.0,250,Laptop,North
1009,2023-01-10,James Wood,1200.0,Laptop,EAST,null


In [0]:
clean_df=clean_df.withColumn(
    "Region",
    initcap(trim(col("Region"))).cast("string")
)

In [0]:
from pyspark.sql.functions import when
clean_df=clean_df.withColumn(
    "Product",
    when(
        col("Product").isin("000","250","200"),"UNKNOWN"
    ).otherwise(col("Product"))
)

In [0]:
from pyspark.sql.functions import when 
clean_df=clean_df.withColumn(
    "Region",
    when(
        col("Region").isin("Laptop"),
        "UNKOWN"
    ).otherwise(col("Region"))
)

In [0]:
display(clean_df)

OrderID,Date,SalesPerson,Amount,Product,Region,Column1
1001,2023-01-05,Alice Smith,1.0,UNKNOWN,UNKOWN,North
1002,2023-01-06,Bob Jones,700.0,Phone,South,null
1003,2023-01-07,Carol Wang,null,Tablet,East,null
1004,2023-01-07,Alice Smith,1100.0,Laptop,North,null
1005,2023-01-08,Bob Jones,700.0,Phone,South,null
1006,2023-01-08,null,850.0,Tablet,West,null
1006,2023-01-08,null,850.0,Tablet,West,null
1007,null,Scott Rogers,900.0,Phone,West,null
1008,2023-01-08,Linda Moore,1.0,UNKNOWN,UNKOWN,North
1009,2023-01-10,James Wood,1200.0,Laptop,East,null


In [0]:
clean_df.select("Column1").distinct().show()
clean_df.filter(col("Column1").isNotNull()).count()

+-------+
|Column1|
+-------+
|  North|
|  NORTH|
|   NULL|
|   West|
+-------+



5

In [0]:
clean_df.select(
    "OrderID",
    "Region",
    "Column1"
).show(50, truncate=False)

+-------+------+-------+
|OrderID|Region|Column1|
+-------+------+-------+
|1001   |UNKOWN|North  |
|1002   |South |NULL   |
|1003   |East  |NULL   |
|1004   |North |NULL   |
|1005   |South |NULL   |
|1006   |West  |NULL   |
|1006   |West  |NULL   |
|1007   |West  |NULL   |
|1008   |UNKOWN|North  |
|1009   |East  |NULL   |
|1010   |South |NULL   |
|1011   |South |NULL   |
|1012   |East  |NULL   |
|1013   |North |NULL   |
|1014   |West  |NULL   |
|1015   |UNKOWN|NORTH  |
|1016   |South |NULL   |
|1017   |West  |NULL   |
|1018   |South |NULL   |
|1019   |East  |NULL   |
|1020   |North |NULL   |
|1021   |South |NULL   |
|1022   |East  |NULL   |
|1023   |West  |NULL   |
|1024   |UNKOWN|NORTH  |
|1025   |South |NULL   |
|1026   |West  |NULL   |
|1026   |West  |NULL   |
|1027   |North |NULL   |
|1028   |East  |NULL   |
|1029   |South |NULL   |
|1030   |North |NULL   |
|1031   |West  |NULL   |
|1032   |East  |NULL   |
|1033   |South |NULL   |
|1034   |UNKOWN|West   |
+-------+------+-------+


In [0]:
from pyspark.sql.functions import when, trim, initcap, col
clean_df=clean_df.withColumn(
    "Region",
    when(
        col("Region").isNull() | (col("Region")=="Unkown") ,
        initcap(trim(col("column1")))
     ).otherwise(col("Region"))
)


In [0]:
clean_df.select("OrderID","Region").show()

+-------+------+
|OrderID|Region|
+-------+------+
|   1001| North|
|   1002| South|
|   1003|  East|
|   1004| North|
|   1005| South|
|   1006|  West|
|   1006|  West|
|   1007|  West|
|   1008| North|
|   1009|  East|
|   1010| South|
|   1011| South|
|   1012|  East|
|   1013| North|
|   1014|  West|
|   1015| North|
|   1016| South|
|   1017|  West|
|   1018| South|
|   1019|  East|
+-------+------+
only showing top 20 rows


In [0]:
clean_df.show(50,truncate=False)


+-------+----------+------------+------+-------+------+
|OrderID|Date      |SalesPerson |Amount|Product|Region|
+-------+----------+------------+------+-------+------+
|1001   |2023-01-05|Alice Smith |1.0   |UNKNOWN|North |
|1002   |2023-01-06|Bob Jones   |700.0 |Phone  |South |
|1003   |2023-01-07|Carol Wang  |NULL  |Tablet |East  |
|1004   |2023-01-07|Alice Smith |1100.0|Laptop |North |
|1005   |2023-01-08|Bob Jones   |700.0 |Phone  |South |
|1006   |2023-01-08|NULL        |850.0 |Tablet |West  |
|1006   |2023-01-08|NULL        |850.0 |Tablet |West  |
|1007   |NULL      |Scott Rogers|900.0 |Phone  |West  |
|1008   |2023-01-08|Linda Moore |1.0   |UNKNOWN|North |
|1009   |2023-01-10|James Wood  |1200.0|Laptop |East  |
|1010   |2023-11-01|Alicia Keys |NULL  |Tablet |South |
|1011   |2023-01-12|Bob Jones   |700.0 |Phone  |South |
|1012   |2023-01-15|Carol Wang  |820.0 |Tablet |East  |
|1013   |2023-01-16|James Wood  |1300.0|Laptop |North |
|1014   |2023-01-17|Anna Bell   |650.0 |Phone  |

In [0]:
clean_df.groupBy("OrderID") \
    .count()\
    .filter(col("count")>1) \
    .show()

+-------+-----+
|OrderID|count|
+-------+-----+
+-------+-----+



In [0]:
clean_df.filter(col("OrderID").isin(1026,1006)).show(50,truncate=False)

+-------+----------+-----------+------+-------+------+
|OrderID|Date      |SalesPerson|Amount|Product|Region|
+-------+----------+-----------+------+-------+------+
|1026   |2023-01-30|Anna Bell  |650.0 |Phone  |West  |
|1006   |2023-01-08|NULL       |850.0 |Tablet |West  |
+-------+----------+-----------+------+-------+------+



In [0]:
clean_df=clean_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import col, when 
clean_df.select([
    col(c).isNull().cast("int").alias(c)
    for c in clean_df.columns
]).show()

+-------+----+-----------+------+-------+------+
|OrderID|Date|SalesPerson|Amount|Product|Region|
+-------+----+-----------+------+-------+------+
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   1|          0|     0|      0|     0|
|      0|   1|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   1|          0|     0|      0|     0|
|      0|   0|          1|     1|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     1|      0|     0|
|      0|   0|          0|     0|      0|     0|
|      0|   0|          0|     1|      0|     0|
|      0|   1|          0|     0|      0|     0|
|      0|   0|      

In [0]:
from pyspark.sql.functions import col, sum

clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in clean_df.columns
]).show()

+-------+----+-----------+------+-------+------+
|OrderID|Date|SalesPerson|Amount|Product|Region|
+-------+----+-----------+------+-------+------+
|      0|   6|          4|     4|      0|     0|
+-------+----+-----------+------+-------+------+



In [0]:
clean_df=clean_df.withColumn(
    "SalesPerson",
    when(
        col("SalesPerson").isNull(),"UNKNOWN"
    ).otherwise(col("SalesPerson"))
)

In [0]:
clean_df.select("SalesPerson").show(50,truncate=False)

+------------+
|SalesPerson |
+------------+
|Alice Smith |
|Bob Jones   |
|Alice Smith |
|James Wood  |
|Anna Bell   |
|Caroline Li |
|Anna Bell   |
|Bob Jones   |
|Scott Rogers|
|Anna Bell   |
|Scott Rogers|
|UNKNOWN     |
|Carol Wang  |
|Carol Wang  |
|Scott Rogers|
|Alicia Keys |
|Bob Jones   |
|Bob Jones   |
|Linda Moore |
|Linda Moore |
|Alice Smith |
|Anna Bell   |
|Bob Jones   |
|Carol Wang  |
|UNKNOWN     |
|James Wood  |
|James Wood  |
|UNKNOWN     |
|Linda Moore |
|Alicia Keys |
|UNKNOWN     |
|Bob Jones   |
|Bob Jones   |
|Bob Jones   |
+------------+



In [0]:
clean_df.select("Amount").show(50,truncate=False)

+------+
|Amount|
+------+
|1.0   |
|700.0 |
|1.0   |
|1.0   |
|650.0 |
|750.0 |
|650.0 |
|690.0 |
|880.0 |
|830.0 |
|900.0 |
|NULL  |
|900.0 |
|NULL  |
|950.0 |
|NULL  |
|700.0 |
|700.0 |
|1150.0|
|1.0   |
|1100.0|
|690.0 |
|700.0 |
|820.0 |
|850.0 |
|1200.0|
|1300.0|
|830.0 |
|NULL  |
|900.0 |
|720.0 |
|700.0 |
|800.0 |
|1.0   |
+------+



In [0]:
clean_df.filter(
    col("Date").isNull()
).select(
    "OrderID",
    "Date",
    "SalesPerson",
    "Amount",
    "Product",
    "Region"
).show(50, truncate=False)

+-------+----+------------+------+-------+------+
|OrderID|Date|SalesPerson |Amount|Product|Region|
+-------+----+------------+------+-------+------+
|1015   |NULL|Alice Smith |1.0   |UNKNOWN|North |
|1024   |NULL|James Wood  |1.0   |UNKNOWN|North |
|1007   |NULL|Scott Rogers|900.0 |Phone  |West  |
|1030   |NULL|Bob Jones   |700.0 |Phone  |North |
|1025   |NULL|Alicia Keys |900.0 |Tablet |South |
|1022   |NULL|Bob Jones   |800.0 |Tablet |East  |
+-------+----+------------+------+-------+------+



In [0]:
clean_df = clean_df.filter(
    col("Date").isNotNull()
)

In [0]:
print("Rows after removing missing Date:", clean_df.count())

Rows after removing missing Date: 24


In [0]:
from pyspark.sql.functions import col, sum

clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in clean_df.columns
]).show()

+-------+----+-----------+------+-------+------+
|OrderID|Date|SalesPerson|Amount|Product|Region|
+-------+----+-----------+------+-------+------+
|      0|   0|          0|     0|      0|     0|
+-------+----+-----------+------+-------+------+



In [0]:
clean_df.groupBy("OrderID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-------+-----+
|OrderID|count|
+-------+-----+
+-------+-----+

